In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 1.26.4


In [4]:
# Define the location of the cleaned dataset
DATA_FILE = "../data/clean/transactions_clean.parquet"

# Load the dataset
transactions = pd.read_parquet(DATA_FILE)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [5]:
# Dataset dimensions
print("Number of rows:", transactions.shape[0])
print("Number of columns:", transactions.shape[1])

# Column names
print("\nColumn names:")
print(transactions.columns.tolist())

# Data types
print("\nData types:")
print(transactions.dtypes)

# First five records
display(transactions.head())

Number of rows: 1041449
Number of columns: 13

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Sheet', 'is_exact_duplicate', 'is_cancellation', 'is_product', 'line_type']

Data types:
Invoice                       object
StockCode                     object
Description           string[python]
Quantity                       int64
InvoiceDate           datetime64[ns]
Price                        float64
Customer ID                    Int64
Country                       object
Sheet                         object
is_exact_duplicate              bool
is_cancellation                 bool
is_product                      bool
line_type                     object
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sheet,is_exact_duplicate,is_cancellation,is_product,line_type
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,False,True,product
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,False,True,product
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,False,True,product
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,False,True,product
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,False,True,product


In [6]:
# Check missing values
print("Missing values:")
display(transactions.isnull().sum())

# Check number of unique customers
print("\nUnique identified customers:")
print(transactions["Customer ID"].nunique())

# Check missing customer IDs
print("\nMissing customer IDs:")
print(transactions["Customer ID"].isnull().sum())

# Check duplicate rows
print("\nExact duplicate rows:")
print(transactions.duplicated().sum())

Missing values:


Invoice                    0
StockCode                  0
Description             1642
Quantity                   0
InvoiceDate                0
Price                      0
Customer ID           231888
Country                    0
Sheet                      0
is_exact_duplicate         0
is_cancellation            0
is_product                 0
line_type                  0
dtype: int64


Unique identified customers:
5942

Missing customer IDs:
231888

Exact duplicate rows:
811


In [7]:
# Basic dataset validation

assert transactions.shape[0] == 1041449
assert transactions.shape[1] == 13

assert transactions["Customer ID"].nunique() == 5942

assert transactions["is_cancellation"].notna().all()
assert transactions["line_type"].notna().all()

assert transactions["Customer ID"].isna().sum() == 231888

print("All basic validation checks passed!")

All basic validation checks passed!


In [9]:
# Calculate the value of each transaction line
transactions["line_value"] = (
    transactions["Quantity"] * transactions["Price"]
)

# Summarise transaction types
type_summary = (
    transactions
    .groupby("line_type")
    .agg(
        Rows=("Invoice", "size"),
        Unique_invoices=("Invoice", "nunique"),
        Net_value=("line_value", "sum")
    )
    .sort_values("Rows", ascending=False)
)

# Calculate percentage of total rows
type_summary["Row_share"] = (
    type_summary["Rows"] / len(transactions) * 100
)

display(
    type_summary.style.format({
        "Net_value": "£{:,.2f}",
        "Row_share": "{:.2f}%"
    })
)

# Verify that all four categories are present
assert set(transactions["line_type"].unique()) == {
    "product", "adjustment", "postage", "other"
}

# Verify that the categories cover every row
assert type_summary["Rows"].sum() == len(transactions)

print("Transaction-type validation passed.")

,Rows,Unique_invoices,Net_value,Row_share
line_type,,,,
product,1035625,48763,"£18,981,298.10",99.44%
postage,3788,3783,"£433,410.51",0.36%
adjustment,1646,1509,"£-88,248.91",0.16%
other,390,313,"£-269,083.50",0.04%


Transaction-type validation passed.


In [10]:
# Isolate cancellation transactions
cancellations = transactions[
    transactions["is_cancellation"] == True
].copy()

# Summarise cancellations by transaction type
cancellation_summary = (
    cancellations
    .groupby("line_type")
    .agg(
        Rows=("Invoice", "size"),
        Unique_invoices=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique"),
        Net_value=("line_value", "sum")
    )
    .sort_values("Rows", ascending=False)
)

display(
    cancellation_summary.style.format({
        "Net_value": "£{:,.2f}"
    })
)

# Overall cancellation checks
print("Total cancellation rows:", len(cancellations))
print("Unique cancellation invoices:", cancellations["Invoice"].nunique())
print("Cancellation rows with customer ID:",
      cancellations["Customer ID"].notna().sum())
print("Cancellation rows without customer ID:",
      cancellations["Customer ID"].isna().sum())

print("Cancellation net value:",
      f"£{cancellations['line_value'].sum():,.2f}")

print("Negative quantity rows:",
      (cancellations["Quantity"] < 0).sum())

print("Zero or positive quantity rows:",
      (cancellations["Quantity"] >= 0).sum())

# Validation
assert len(cancellations) == 19165
assert cancellations["Invoice"].str.startswith("C").all()

print("\nCancellation validation passed.")

,Rows,Unique_invoices,Customers,Net_value
line_type,,,,
product,17973,7405,2444,"£-719,656.34"
adjustment,734,671,280,"£-437,905.72"
postage,238,233,160,"£-15,552.02"
other,220,151,10,"£-292,189.58"


Total cancellation rows: 19165
Unique cancellation invoices: 8292
Cancellation rows with customer ID: 18446
Cancellation rows without customer ID: 719
Cancellation net value: £-1,465,303.66
Negative quantity rows: 19164
Zero or positive quantity rows: 1

Cancellation validation passed.


In [11]:
# Find cancellation rows with zero or positive quantity
exceptional_cancellations = cancellations[
    cancellations["Quantity"] >= 0
]

display(
    exceptional_cancellations[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "line_type",
            "line_value"
        ]
    ]
)

,Invoice,StockCode,Description,Quantity,Price,Customer ID,line_type,line_value
76581,C496350,M,Manual,1,373.57,<NA>,adjustment,373.57


In [12]:
# Check customer consistency at invoice level
invoice_check = (
    transactions
    .groupby("Invoice")
    .agg(
        unique_customers=("Customer ID", "nunique"),
        missing_customer_rows=("Customer ID", lambda x: x.isna().sum()),
        total_rows=("Invoice", "size")
    )
)

print("Total unique invoices:", len(invoice_check))

print(
    "Invoices linked to multiple customers:",
    (invoice_check["unique_customers"] > 1).sum()
)

print(
    "Invoices with both identified and missing customer rows:",
    (
        (invoice_check["unique_customers"] == 1) &
        (invoice_check["missing_customer_rows"] > 0)
    ).sum()
)

print(
    "Invoices without any Customer ID:",
    (invoice_check["unique_customers"] == 0).sum()
)

# Display any invoices associated with multiple customers
display(
    invoice_check[
        invoice_check["unique_customers"] > 1
    ].head(10)
)

Total unique invoices: 50229
Invoices linked to multiple customers: 0
Invoices with both identified and missing customer rows: 0
Invoices without any Customer ID: 5353


,unique_customers,missing_customer_rows,total_rows
Invoice,,,


In [13]:
# Identify valid purchase transactions
valid_purchase_rows = transactions[
    transactions["Customer ID"].notna()
    & (~transactions["is_cancellation"])
    & (transactions["line_type"] == "product")
].copy()

# Create one record per customer purchase invoice
purchase_invoices = (
    valid_purchase_rows
    .groupby(["Customer ID", "Invoice"])
    .agg(
        purchase_date=("InvoiceDate", "min"),
        product_rows=("Invoice", "size"),
        purchase_value=("line_value", "sum")
    )
    .reset_index()
)

print("Valid purchase rows:", len(valid_purchase_rows))
print("Unique purchase invoices:", len(purchase_invoices))
print("Unique customers:", purchase_invoices["Customer ID"].nunique())

print(
    "Purchase invoices with non-positive product value:",
    (purchase_invoices["purchase_value"] <= 0).sum()
)

display(purchase_invoices.head())

Valid purchase rows: 788249
Unique purchase invoices: 36597
Unique customers: 5853
Purchase invoices with non-positive product value: 3


,Customer ID,Invoice,purchase_date,product_rows,purchase_value
0,12346,499763,2010-03-02 13:08:00,5,27.05
1,12346,513774,2010-06-28 13:53:00,19,142.31
2,12346,541431,2011-01-18 10:01:00,1,77183.60
3,12347,529924,2010-10-31 14:20:00,40,611.53
4,12347,537626,2010-12-07 14:57:00,31,711.79


In [14]:
# Identify the three exceptional invoices
exceptional_invoices = purchase_invoices[
    purchase_invoices["purchase_value"] <= 0
]

display(exceptional_invoices)

# Inspect their underlying product rows
exceptional_details = valid_purchase_rows[
    valid_purchase_rows["Invoice"].isin(
        exceptional_invoices["Invoice"]
    )
][
    [
        "Customer ID",
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "line_value"
    ]
].sort_values(["Invoice", "StockCode"])

display(exceptional_details)

,Customer ID,Invoice,purchase_date,product_rows,purchase_value
5768,13256,578841,2011-11-25 15:57:00,1,0.0
14359,14646,564651,2011-08-26 14:19:00,4,0.0
31778,17560,543599,2011-02-10 13:08:00,1,0.0


,Customer ID,Invoice,StockCode,Description,Quantity,Price,line_value
587476,17560,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,0.0,0.0
814713,14646,564651,21786,POLKADOT RAIN HAT,144,0.0,0.0
814712,14646,564651,22955,36 FOIL STAR CAKE CASES,144,0.0,0.0
814711,14646,564651,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,0.0,0.0
814710,14646,564651,23270,SET OF 2 CERAMIC PAINTED HEARTS,96,0.0,0.0
1001707,13256,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.0,0.0


In [15]:
# All customers present in the cleaned dataset
all_customers = set(
    transactions["Customer ID"].dropna().unique()
)

# Customers with at least one valid purchase
purchasing_customers = set(
    purchase_invoices["Customer ID"].unique()
)

# Customers without a valid purchase
customers_without_purchases = (
    all_customers - purchasing_customers
)

print("Total identified customers:", len(all_customers))
print("Customers with valid purchases:", len(purchasing_customers))
print("Customers without valid purchases:",
      len(customers_without_purchases))

# Examine their transaction types
non_purchase_customer_rows = transactions[
    transactions["Customer ID"].isin(
        customers_without_purchases
    )
]

display(
    non_purchase_customer_rows
    .groupby("line_type")
    .agg(
        Rows=("Invoice", "size"),
        Unique_invoices=("Invoice", "nunique"),
        Net_value=("line_value", "sum")
    )
)

Total identified customers: 5942
Customers with valid purchases: 5853
Customers without valid purchases: 89


,Rows,Unique_invoices,Net_value
line_type,,,
adjustment,112,112,-78416.78
other,6,6,-90.00
postage,2,2,241.00
product,87,25,-1406.13


In [16]:
# Extract product transactions belonging to the 89 customers
non_purchase_products = non_purchase_customer_rows[
    non_purchase_customer_rows["line_type"] == "product"
].copy()

# Summarise at invoice level
invoice_investigation = (
    non_purchase_products
    .groupby(["Customer ID", "Invoice"])
    .agg(
        invoice_date=("InvoiceDate", "min"),
        rows=("Invoice", "size"),
        total_quantity=("Quantity", "sum"),
        net_value=("line_value", "sum"),
        is_cancellation=("is_cancellation", "first")
    )
    .reset_index()
    .sort_values(["Customer ID", "invoice_date"])
)

display(invoice_investigation)

# Check the nature of these invoices
print("Total invoices:", len(invoice_investigation))
print("Cancellation invoices:",
      invoice_investigation["is_cancellation"].sum())
print("Invoices with negative quantity:",
      (invoice_investigation["total_quantity"] < 0).sum())
print("Invoices with zero quantity:",
      (invoice_investigation["total_quantity"] == 0).sum())
print("Invoices with positive quantity:",
      (invoice_investigation["total_quantity"] > 0).sum())
print("Invoices with negative value:",
      (invoice_investigation["net_value"] < 0).sum())
print("Invoices with zero value:",
      (invoice_investigation["net_value"] == 0).sum())

,Customer ID,Invoice,invoice_date,rows,total_quantity,net_value,is_cancellation
0,12896,C491037,2009-12-09 09:28:00,1,-5,-29.75,True
1,13342,C507204,2010-05-06 16:44:00,2,-4,-14.10,True
2,13353,C533099,2010-11-16 11:14:00,3,-3,-28.85,True
3,13749,C493864,2010-01-07 16:52:00,2,-7,-10.25,True
4,14120,C525539,2010-10-05 19:12:00,1,-4,-23.80,True
5,14337,C490942,2009-12-08 14:04:00,36,-393,-658.63,True
6,14763,C490743,2009-12-07 17:59:00,1,-1,-9.95,True
7,14864,C490987,2009-12-08 17:07:00,2,-17,-48.75,True
8,14914,C504358,2010-04-13 10:14:00,1,-3,-5.85,True
9,14925,C490931,2009-12-08 13:44:00,1,-1,-7.95,True


Total invoices: 25
Cancellation invoices: 25
Invoices with negative quantity: 25
Invoices with zero quantity: 0
Invoices with positive quantity: 0
Invoices with negative value: 25
Invoices with zero value: 0


In [17]:
# Set the analysis reference date to one day after
# the final transaction date
snapshot_date = (
    transactions["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

# Calculate customer-level purchase metrics
customer_metrics = (
    purchase_invoices
    .groupby("Customer ID")
    .agg(
        last_purchase_date=("purchase_date", "max"),
        frequency=("Invoice", "nunique"),
        product_revenue=("purchase_value", "sum")
    )
    .reset_index()
)

# Calculate recency in days
customer_metrics["recency"] = (
    snapshot_date - customer_metrics["last_purchase_date"]
).dt.days

print("Analysis reference date:", snapshot_date)
print("Customers in metrics table:", len(customer_metrics))
print("Total purchase invoices:", customer_metrics["frequency"].sum())

print("\nRecency statistics:")
display(customer_metrics["recency"].describe())

print("\nFrequency statistics:")
display(customer_metrics["frequency"].describe())

print("\nCustomer metrics preview:")
display(customer_metrics.head(10))

Analysis reference date: 2011-12-10 12:50:00
Customers in metrics table: 5853
Total purchase invoices: 36597

Recency statistics:


count    5853.000000
mean      200.166240
std       208.505959
min         1.000000
25%        25.000000
50%        95.000000
75%       379.000000
max       739.000000
Name: recency, dtype: float64


Frequency statistics:


count    5853.000000
mean        6.252691
std        12.750278
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       373.000000
Name: frequency, dtype: float64


Customer metrics preview:


,Customer ID,last_purchase_date,frequency,product_revenue,recency
0,12346,2011-01-18 10:01:00,3,77352.96,326
1,12347,2011-12-07 15:52:00,8,4921.53,2
2,12348,2011-09-25 13:13:00,5,1658.40,75
3,12349,2011-11-21 09:51:00,3,3678.69,19
4,12350,2011-02-02 16:01:00,1,294.40,310
5,12351,2010-11-29 15:23:00,1,300.93,375
6,12352,2011-11-03 14:37:00,9,1729.54,36
7,12353,2011-05-19 17:47:00,2,406.76,204
8,12354,2011-04-21 13:11:00,1,1079.40,232
9,12355,2011-05-09 13:49:00,2,947.61,214


In [18]:
# Select rows relevant to customer monetary value
monetary_rows = transactions[
    transactions["Customer ID"].notna()
    & transactions["line_type"].isin(["product", "adjustment"])
].copy()

# Calculate net monetary value for each customer
customer_monetary = (
    monetary_rows
    .groupby("Customer ID")
    .agg(
        monetary_value=("line_value", "sum"),
        product_adjustments_rows=("Invoice", "size")
    )
    .reset_index()
)

# Merge monetary value with recency and frequency
customer_metrics = customer_metrics.merge(
    customer_monetary,
    on="Customer ID",
    how="left"
)

# Validate the resulting customer table
print("Customers in final metrics table:", len(customer_metrics))
print("Missing monetary values:",
      customer_metrics["monetary_value"].isna().sum())

print("\nMonetary value statistics:")
display(customer_metrics["monetary_value"].describe())

print("\nCustomer metrics preview:")
display(
    customer_metrics[
        [
            "Customer ID",
            "recency",
            "frequency",
            "product_revenue",
            "monetary_value"
        ]
    ].head(10)
)

# Identify customers with negative net monetary value
print(
    "\nCustomers with negative monetary value:",
    (customer_metrics["monetary_value"] < 0).sum()
)

print(
    "Customers with zero monetary value:",
    (customer_metrics["monetary_value"] == 0).sum()
)

Customers in final metrics table: 5853
Missing monetary values: 0

Monetary value statistics:


count      5853.000000
mean       2784.283716
std       13722.450852
min       -3767.200000
25%         331.260000
50%         840.080000
75%        2172.470000
max      570380.610000
Name: monetary_value, dtype: float64


Customer metrics preview:


,Customer ID,recency,frequency,product_revenue,monetary_value
0,12346,326,3,77352.96,-263.68
1,12347,2,8,4921.53,4921.53
2,12348,75,5,1658.40,1658.40
3,12349,19,3,3678.69,3654.54
4,12350,310,1,294.40,294.40
5,12351,375,1,300.93,300.93
6,12352,36,9,1729.54,1609.21
7,12353,204,2,406.76,406.76
8,12354,232,1,1079.40,1079.40
9,12355,214,2,947.61,947.61



Customers with negative monetary value: 10
Customers with zero monetary value: 11


In [19]:
# Identify customers with zero or negative monetary value
unusual_customers = customer_metrics[
    customer_metrics["monetary_value"] <= 0
].copy()

print("Customers with zero/negative value:",
      len(unusual_customers))

display(
    unusual_customers[
        [
            "Customer ID",
            "recency",
            "frequency",
            "product_revenue",
            "monetary_value"
        ]
    ].sort_values("monetary_value")
)

# Investigate Customer 12346
customer_12346 = transactions[
    (transactions["Customer ID"] == 12346)
    & (transactions["line_type"].isin(["product", "adjustment"]))
].copy()

print("\nCustomer 12346 — transaction summary")

display(
    customer_12346
    .groupby(["line_type", "is_cancellation"])
    .agg(
        rows=("Invoice", "size"),
        invoices=("Invoice", "nunique"),
        net_value=("line_value", "sum")
    )
)

# Inspect invoice-level values
customer_12346_invoices = (
    customer_12346
    .groupby(["Invoice", "is_cancellation"])
    .agg(
        invoice_date=("InvoiceDate", "min"),
        rows=("Invoice", "size"),
        net_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("net_value")
)

display(customer_12346_invoices)

Customers with zero/negative value: 21


,Customer ID,recency,frequency,product_revenue,monetary_value
1688,14063,687,7,9471.50,-3.767200e+03
3016,15413,692,5,6798.72,-2.531380e+03
0,12346,326,3,77352.96,-2.636800e+02
154,12503,584,1,896.00,-2.300000e+02
732,13091,28,2,1108.80,-1.302200e+02
4046,16454,64,2,7.45,-6.600000e+00
752,13112,541,1,20.60,-5.440000e+00
1836,14213,405,1,1192.20,-1.421085e-14
5637,18072,186,1,247.44,-5.329071e-15
3538,15940,311,1,35.80,-3.552714e-15



Customer 12346 — transaction summary


rows  invoices  net_value
line_type  is_cancellation                           
adjustment True               12         3    -433.04
product    False              25         3   77352.96
           True                1         1  -77183.60

,Invoice,is_cancellation,invoice_date,rows,net_value
6,C541433,True,2011-01-18 10:17:00,1,-77183.60
4,C514024,True,2010-06-30 11:22:00,10,-328.54
3,C495800,True,2010-01-26 17:27:00,1,-103.50
5,C525099,True,2010-10-04 09:54:00,1,-1.00
0,499763,False,2010-03-02 13:08:00,5,27.05
1,513774,False,2010-06-28 13:53:00,19,142.31
2,541431,False,2011-01-18 10:01:00,1,77183.60


In [20]:
# Round monetary values to two decimal places
customer_metrics["monetary_value"] = (
    customer_metrics["monetary_value"].round(2)
)

# Recheck unusual values
print(
    "Negative monetary values:",
    (customer_metrics["monetary_value"] < 0).sum()
)

print(
    "Zero monetary values:",
    (customer_metrics["monetary_value"] == 0).sum()
)

print(
    "Missing monetary values:",
    customer_metrics["monetary_value"].isna().sum()
)

Negative monetary values: 7
Zero monetary values: 20
Missing monetary values: 0


In [21]:
# Create the final RFM table
rfm = customer_metrics[
    [
        "Customer ID",
        "recency",
        "frequency",
        "monetary_value"
    ]
].copy()

# Sort by customer ID
rfm = rfm.sort_values("Customer ID").reset_index(drop=True)

# Validation checks
print("RFM table dimensions:", rfm.shape)
print("Unique customers:", rfm["Customer ID"].nunique())

print("\nMissing values:")
print(rfm.isna().sum())

print("\nMetric validation:")
print("Customers with recency below 1:",
      (rfm["recency"] < 1).sum())

print("Customers with frequency below 1:",
      (rfm["frequency"] < 1).sum())

print("Total purchase invoices:", rfm["frequency"].sum())

print("Duplicate customer records:",
      rfm["Customer ID"].duplicated().sum())

# Reconcile monetary value against source transactions
reconciled_value = (
    monetary_rows[
        monetary_rows["Customer ID"].isin(rfm["Customer ID"])
    ]["line_value"].sum()
)

print("\nMonetary reconciliation:")
print("RFM monetary total:", round(rfm["monetary_value"].sum(), 2))
print("Source transaction total:", round(reconciled_value, 2))

# Final assertions
assert rfm.shape == (5853, 4)
assert rfm["Customer ID"].nunique() == 5853
assert rfm.isna().sum().sum() == 0
assert (rfm["recency"] >= 1).all()
assert (rfm["frequency"] >= 1).all()
assert rfm["Customer ID"].duplicated().sum() == 0
assert rfm["frequency"].sum() == 36597
assert round(rfm["monetary_value"].sum(), 2) == round(reconciled_value, 2)

print("\nAll RFM validation assertions passed!")

display(rfm.head(10))

RFM table dimensions: (5853, 4)
Unique customers: 5853

Missing values:
Customer ID       0
recency           0
frequency         0
monetary_value    0
dtype: int64

Metric validation:
Customers with recency below 1: 0
Customers with frequency below 1: 0
Total purchase invoices: 36597
Duplicate customer records: 0

Monetary reconciliation:
RFM monetary total: 16296412.59
Source transaction total: 16296412.59

All RFM validation assertions passed!


,Customer ID,recency,frequency,monetary_value
0,12346,326,3,-263.68
1,12347,2,8,4921.53
2,12348,75,5,1658.40
3,12349,19,3,3654.54
4,12350,310,1,294.40
5,12351,375,1,300.93
6,12352,36,9,1609.21
7,12353,204,2,406.76
8,12354,232,1,1079.40
9,12355,214,2,947.61


In [22]:
# Compare customer populations
all_customer_ids = set(
    transactions["Customer ID"].dropna().unique()
)

rfm_customer_ids = set(rfm["Customer ID"].unique())

# Customers without qualifying purchases
excluded_customer_ids = all_customer_ids - rfm_customer_ids

print("Original identified customers:", len(all_customer_ids))
print("Customers in RFM table:", len(rfm_customer_ids))
print("Customers excluded from RFM:", len(excluded_customer_ids))

# Check whether excluded customers have valid purchase invoices
excluded_purchase_count = purchase_invoices[
    purchase_invoices["Customer ID"].isin(excluded_customer_ids)
].shape[0]

print("Purchase invoices belonging to excluded customers:",
      excluded_purchase_count)

# Check whether every RFM customer has a purchase invoice
missing_purchase_customers = (
    rfm_customer_ids - purchasing_customers
)

print("RFM customers without purchase records:",
      len(missing_purchase_customers))

# Assertions
assert len(all_customer_ids) == 5942
assert len(rfm_customer_ids) == 5853
assert len(excluded_customer_ids) == 89
assert excluded_purchase_count == 0
assert len(missing_purchase_customers) == 0
assert rfm_customer_ids.issubset(all_customer_ids)

print("\nCustomer coverage validation passed!")

Original identified customers: 5942
Customers in RFM table: 5853
Customers excluded from RFM: 89
Purchase invoices belonging to excluded customers: 0
RFM customers without purchase records: 0

Customer coverage validation passed!
